In [ ]:
!pip install transformers datasets gradio torch scikit-learn rouge-score
!pip install sentence-transformers


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=7802342e2d71ea83e7059fa68fea2435cc0773e27935a71e267c3547991b263f
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


Bu hücre, proje için gerekli Python kütüphanelerini (`transformers`, `datasets`, `gradio`, `torch`, `scikit-learn`, `rouge-score` ve `sentence-transformers` dahil) yükler.

In [ ]:
from datasets import load_dataset

# Veri setini indir
dataset = load_dataset("truehealth/medicationqa")

# Genel yapıya bak
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/541 [00:00<?, ?B/s]

data/train-00000-of-00001-7427a10e891759(…):   0%|          | 0.00/215k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/690 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Question', 'Focus (Drug)', 'Question Type', 'Answer', 'Section Title', 'URL'],
        num_rows: 690
    })
})


Bu hücre, `load_dataset` fonksiyonunu kullanarak "truehealth/medicationqa" veri setini Hugging Face Hub'dan yükler ve yapısını yazdırır. Ayrıca, HF Hub'a kimlik doğrulanmamış istekler hakkında bir uyarı göstererek bir `HF_TOKEN` ayarlanmasını önerir.

In [ ]:
import pandas as pd

# Dataset'i pandas'a çevir
df = pd.DataFrame(dataset['train'])

# İlk 5 satıra bak
print(df.head())

# Sütun bilgileri
print("\nSütunlar:", df.columns.tolist())
print("Toplam satır:", len(df))

# Eksik değerlere bak
print("\nEksik değerler:")
print(df.isnull().sum())

                                            Question   Focus (Drug)  \
0  how does rivatigmine and otc sleep medicine in...   rivastigmine   
1                   how does valium affect the brain         Valium   
2                                   what is morphine       morphine   
3            what are the milligrams for oxycodone e   oxycodone ER   
4     81% aspirin contain resin and shellac in it. ?  aspirin 81 mg   

  Question Type                                             Answer  \
0   Interaction  tell your doctor and pharmacist what prescript...   
1        Action  Diazepam is a benzodiazepine that exerts anxio...   
2   Information  Morphine is a pain medication of the opiate fa...   
3          Dose                … 10 mg … 20 mg … 40 mg … 80 mg ...   
4    Ingredient              Inactive Ingredients  Ingredient Name   

                               Section Title  \
0  What special precautions should I follow?   
1                      CLINICAL PHARMACOLOGY   
2       

Bu hücre, yüklenen veri setinin 'train' bölümünü bir pandas DataFrame'e dönüştürür. Ardından DataFrame'in ilk 5 satırını görüntüler, sütun adlarını listeler, toplam satır sayısını gösterir ve her sütundaki eksik değerleri kontrol eder.

In [ ]:
from sklearn.model_selection import train_test_split

# Sadece gerekli sütunları al
df = df[['Question', 'Focus (Drug)', 'Question Type', 'Answer']]

# Answer ve Question'da eksik olan satırları sil
df = df.dropna(subset=['Question', 'Answer'])

# Boş string olanları temizle
df = df[df['Question'].str.strip() != '']
df = df[df['Answer'].str.strip() != '']

# Indexi sıfırla
df = df.reset_index(drop=True)

print("Temizlendikten sonra satır sayısı:", len(df))

# Train / Validation / Test olarak böl
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=42)

print("Eğitim seti:", len(train_df))
print("Doğrulama seti:", len(val_df))
print("Test seti:", len(test_df))

Temizlendikten sonra satır sayısı: 689
Eğitim seti: 482
Doğrulama seti: 103
Test seti: 104


Bu hücre, veri temizleme ve bölme işlemleri yapar. Belirli sütunları (`Question`, `Focus (Drug)`, `Question Type`, `Answer`) seçer, 'Question' veya 'Answer' sütunlarında eksik değeri olan satırları kaldırır ve 'Question' veya 'Answer' sütunlarında boş dize olan satırları siler. Son olarak, temizlenmiş veriyi eğitim, doğrulama ve test setlerine ayırır ve her setin boyutunu yazdırır.

In [ ]:
from sentence_transformers import SentenceTransformer

# Modelin initialize edilmesi fine-tuning için gereklidir.
# Ancak, fine-tuned model kullanıldığı için initial embedding'lerin kaydedilmesine gerek yoktur.
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# train_questions ve train_answers, AY9gK_YOoF57 hücresinde df'den yeniden türetildiği için
# burada ayrıca tanımlanmasına ve kaydedilmesine gerek yoktur.
# Bu yorum satırlarının altındaki tüm kod kaldırılmıştır.

# print("Sorular encode ediliyor...")
# train_questions = train_df['Question'].tolist()
# train_answers    = train_df['Answer'].tolist()

# train_embeddings = model.encode(train_questions, show_progress_bar=True)
# print("Tamamlandı! Embedding shape:", train_embeddings.shape)

# with open('train_embeddings.pkl', 'wb') as f:
#     pickle.dump(train_embeddings, f)

# with open('train_data.pkl', 'wb') as f:
#     pickle.dump({'questions': train_questions, 'answers': train_answers}, f)

# print("Kaydedildi!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Bu hücre, `sentence-transformers/all-MiniLM-L6-v2` kullanarak bir `SentenceTransformer` modeli başlatır. Ayrıca, soruları ve cevapları kodlamak ve kaydetmek için kullanılacak olan yorum satırı halindeki kodları içerir, ancak `model` daha sonra ince ayarlandığı için buna artık gerek yoktur.

In [ ]:
# This cell previously contained the pre-fine-tuned chatbot function and an identical normalize_query function.
# The unused chatbot function has been removed, and the normalization function has been consolidated into `normalize_text` in a previous cell.

Bu hücre bir yer tutucudur ve daha önce önceden ince ayarlanmış bir sohbet botu fonksiyonu ile fazlalık bir `normalize_query` fonksiyonu içerdiğini belirtir; bu ikisi de o zamandan beri kaldırılmış veya birleştirilmiştir.

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import nltk
nltk.download('punkt')
import pandas as pd

# This cell originally calculated and printed pre-fine-tuning metrics.
# The user requested to remove this display.

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Bu hücre, değerlendirme metrikleri (BLEU ve ROUGE skorları) için kütüphaneleri içe aktarır ve NLTK'dan 'punkt' tokenleştiricisini indirir. Ayrıca, daha önce ince ayar öncesi metrikleri gösteren kodu içerdiğini, ancak kullanıcı isteği üzerine kaldırıldığını belirtir.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import string # For punctuation removal
import pandas as pd

# Re-define dataset, df, train_df, val_df, test_df to ensure they are available
# Veri setini indir
dataset = load_dataset("truehealth/medicationqa")

# Dataset'i pandas'a çevir
df = pd.DataFrame(dataset['train'])

# Sadece gerekli sütunları al
df = df[['Question', 'Focus (Drug)', 'Question Type', 'Answer']]

# Answer ve Question'da eksik olan satırları sil
df = df.dropna(subset=['Question', 'Answer'])

# Boş string olanları temizle
df = df[df['Question'].str.strip() != '']
df = df[df['Answer'].str.strip() != '']

# Indexi sıfırla
df = df.reset_index(drop=True)

# Train / Validation / Test olarak böl
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=42)

def normalize_text(text):
    # Normalize text by removing punctuation, converting to lowercase and stripping whitespace
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.lower().strip()

def exact_match(referans, tahmin_full_output):
    # Extract actual answer text from the chatbot's formatted output for evaluation
    actual_tahmin_text = ''
    if "Bu soruya uygun bir cevap bulunamadı." in tahmin_full_output:
        actual_tahmin_text = ""
    elif '**Cevap:**' in tahmin_full_output:
        start_index = tahmin_full_output.find('**Cevap:**') + len('**Cevap:**')
        end_index = tahmin_full_output.find('\n\n*Benzerlik Skoru:')
        if end_index != -1 and start_index != -1:
            actual_tahmin_text = tahmin_full_output[start_index:end_index].strip()
        else:
            actual_tahmin_text = tahmin_full_output.strip() # Fallback if format changes
    else:
        actual_tahmin_text = tahmin_full_output.strip()

    if not actual_tahmin_text and not normalize_text(referans):
        return 1 # If both are empty/no answer, consider it a match
    return int(normalize_text(referans) == normalize_text(actual_tahmin_text))

def f1_score_hesapla(referans, tahmin_full_output):
    # Extract actual answer text from the chatbot's formatted output for evaluation
    actual_tahmin_text = ''
    if "Bu soruya uygun bir cevap bulunamadı." in tahmin_full_output:
        actual_tahmin_text = ""
    elif '**Cevap:**' in tahmin_full_output:
        start_index = tahmin_full_output.find('**Cevap:**') + len('**Cevap:**')
        end_index = tahmin_full_output.find('\n\n*Benzerlik Skoru:')
        if end_index != -1 and start_index != -1:
            actual_tahmin_text = tahmin_full_output[start_index:end_index].strip()
        else:
            actual_tahmin_text = tahmin_full_output.strip() # Fallback if format changes
    else:
        actual_tahmin_text = tahmin_full_output.strip()

    referans_tokens = normalize_text(referans).split()
    tahmin_tokens = normalize_text(actual_tahmin_text).split()

    ortak = set(referans_tokens) & set(tahmin_tokens)

    if len(ortak) == 0:
        return 0.0

    # Avoid division by zero if tahmin_tokens is empty
    precision = len(ortak) / len(tahmin_tokens) if len(tahmin_tokens) > 0 else 0.0
    recall = len(ortak) / len(referans_tokens) if len(referans_tokens) > 0 else 0.0

    if (precision + recall) == 0:
        return 0.0
    f1 = 2 * precision * recall / (precision + recall)
    return f1

Bu hücre, gerekli tüm veri çerçevelerinin (`df`, `train_df`, `val_df`, `test_df`) mevcut olduğundan emin olmak için veri kümesi yükleme, temizleme ve bölme adımlarını yeniden tanımlar. Ayrıca, metin ön işleme için `normalize_text` ve sohbet botu yanıtlarını değerlendirmek için `exact_match` ve `f1_score_hesapla` olmak üzere iki yardımcı fonksiyon tanımlar.

### Niyet Sınıflandırması (Intent Classification) Eklenmesi

Kullanıcı sorusunun niyetini sınıflandırmak için yeni bir model eğitelim. Bu model, kullanıcının sorusunun hangi kategoriye (örn. 'Interaction', 'Dose', 'Information') girdiğini tahmin edecek ve sohbet botu çıktısına bu bilgiyi ekleyecektir.

Bu markdown hücresi, sohbet botuna Niyet Sınıflandırması ekleme hedefini tanıtır ve kullanıcı sorgularını 'Etkileşim', 'Doz' veya 'Bilgi' gibi türlere ayırmak için yeni bir modelin eğitileceğini açıklar.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# 'Question Type' sütununu etiket kodlaması yapalım
label_encoder = LabelEncoder()
# Tüm 'Question Type' etiketlerini öğrenmek için encoder'ı tüm veri setine uyarla
label_encoder.fit(df['Question Type'])
train_labels_encoded = label_encoder.transform(train_df['Question Type'])
test_labels_encoded  = label_encoder.transform(test_df['Question Type'])

# Eğitim sorularının embedding'lerini oluşturalım
# `model` (SentenceTransformer) zaten bellekte yüklü ve fine-tuned.
print("Eğitim sorularının embedding'leri oluşturuluyor...")
X_train_intent = model.encode(train_df['Question'].tolist(), show_progress_bar=True)
print("Tamamlandı! Eğitim embedding'lerinin şekli:", X_train_intent.shape)

# Niyet sınıflandırıcıyı eğitelim (örn. Lojistik Regresyon)
intent_classifier = LogisticRegression(max_iter=1000, random_state=42)
intent_classifier.fit(X_train_intent, train_labels_encoded)

print("Niyet sınıflandırıcı eğitildi.")

Eğitim sorularının embedding'leri oluşturuluyor...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Tamamlandı! Eğitim embedding'lerinin şekli: (482, 384)
Niyet sınıflandırıcı eğitildi.


Bu hücre, bir niyet sınıflandırıcı hazırlar ve eğitir. 'Question Type' etiketlerini sayısal formata dönüştürmek için `LabelEncoder` kullanır, eğitim sorularını `SentenceTransformer` modelini kullanarak embedding'lere dönüştürür ve ardından bu embedding'ler ve kodlanmış etiketler üzerinde bir `LogisticRegression` modeli eğitir.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Test soruları için embedding'leri oluşturalım
X_test_intent = model.encode(test_df['Question'].tolist(), show_progress_bar=True)

# Test seti üzerinde niyet sınıflandırıcısının performansını değerlendirelim
y_pred_intent = intent_classifier.predict(X_test_intent)

# test_labels_encoded içinde bulunan benzersiz etiketleri al
unique_test_labels = np.unique(test_labels_encoded)
# Bu etiketlere karşılık gelen sınıf isimlerini label_encoder'dan al
target_names_for_report = label_encoder.inverse_transform(unique_test_labels)

print("\nNiyet Sınıflandırıcısının Doğruluk Skoru:", accuracy_score(test_labels_encoded, y_pred_intent))
print("\nNiyet Sınıflandırma Raporu:")
print(classification_report(test_labels_encoded, y_pred_intent, target_names=target_names_for_report, labels=unique_test_labels))

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Niyet Sınıflandırıcısının Doğruluk Skoru: 0.5769230769230769

Niyet Sınıflandırma Raporu:
                   precision    recall  f1-score   support

           Action       1.00      0.43      0.60         7
      Action/time       0.50      0.67      0.57         3
     Alternatives       0.00      0.00      0.00         2
       Appearance       1.00      1.00      1.00         4
      Brand names       0.00      0.00      0.00         1
       Comparison       0.00      0.00      0.00         1
             Dose       0.86      0.86      0.86        14
        Dose/time       0.00      0.00      0.00         1
       Indication       1.00      0.23      0.38        13
      Information       0.42      0.93      0.58        15
       Ingredient       1.00      0.50      0.67         2
      Interaction       0.50      0.86      0.63         7
         Overdose       0.00      0.00      0.00         1
     Side effects       0.62      0.42      0.50        12
Side effects/time      

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Bu hücre, eğitilmiş niyet sınıflandırıcısının performansını değerlendirir. Test sorularını kodlar, niyet etiketlerini tahmin eder ve ardından doğruluk skorunu ve her niyet kategorisi için kesinlik, geri çağırma ve F1-skoru gösteren ayrıntılı bir sınıflandırma raporunu yazdırır.

## DistilBERT Tabanlı SentenceTransformer Modelini Fine-tuning Etme

Bu markdown hücresi, DistilBERT tabanlı SentenceTransformer modelinin ince ayar bölümünün başlangıcını gösteren bir başlık görevi görür.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.sentence_transformer import losses # DeprecationWarning'i gidermek için güncellendi
from torch.utils.data import DataLoader

# Daha önce kullandığımız modeli yükle (veya sıfırdan bir DistilBERT tabanlı model yükle)
# Mevcut modelimiz zaten bellekte 'model' değişkeninde duruyor.
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2') # Eğer model yeniden yüklenecekse

# Eğitim verilerini InputExample formatına dönüştür
train_examples = []
for i, row in train_df.iterrows():
    train_examples.append(InputExample(texts=[row['Question'], row['Answer']]))

# Dataloader oluştur
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# Kayıp fonksiyonunu tanımla
# MultipleNegativesRankingLoss, verilen soru ve cevap çiftlerini yakınlaştırmayı,
# diğerlerini uzaklaştırmayı hedefler (kontrastif öğrenme).
train_loss = losses.MultipleNegativesRankingLoss(model=model)

print("Veri ve kayıp fonksiyonu fine-tuning için hazırlandı.")

Veri ve kayıp fonksiyonu fine-tuning için hazırlandı.


Bu hücre, SentenceTransformer modelinin ince ayarlanması için verileri hazırlar. Eğitim sorularını ve cevaplarını `InputExample` nesnelerine dönüştürür, toplu işleme için bir `DataLoader` oluşturur ve ince ayar süreci için kayıp fonksiyonu olarak `MultipleNegativesRankingLoss`'u tanımlar.

### Fine-tuning İşlemini Başlatma
Şimdi `model.fit()` metodunu kullanarak fine-tuning işlemini başlatabiliriz. Burada epoch sayısı, öğrenme oranı gibi parametreleri belirleyeceğiz.

Bu markdown hücresi, ince ayar sürecinin başlatılmasını açıklayan bir başlık görevi görür ve `model.fit()`'in belirtilen parametrelerle kullanılacağını vurgular.

In [ ]:
# Fine-tuning parametrelerini tanımla
num_epochs = 3
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1) # %10 warmup steps

print(f"Fine-tuning başlatılıyor: {num_epochs} epoch, warmup steps: {warmup_steps}")

# Modeli eğit
# Bu işlem biraz zaman alabilir ve GPU kullanacaktır.
model.fit(train_objectives=[(train_dataloader, train_loss)],
          epochs=num_epochs,
          warmup_steps=warmup_steps,
          output_path='./fine_tuned_sentence_transformer_model')

print("SentenceTransformer modeli fine-tuning edildi ve kaydedildi!")

# Eğitilmiş modeli tekrar yükleyebilir veya mevcut 'model' değişkenini kullanmaya devam edebilirsiniz.
# fine_tuned_model = SentenceTransformer('./fine_tuned_sentence_transformer_model')


Fine-tuning başlatılıyor: 3 epoch, warmup steps: 9


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SentenceTransformer modeli fine-tuning edildi ve kaydedildi!


Bu hücre, SentenceTransformer modelinin ince ayarını yürütür. Dönem sayısını ve ısınma adımlarını ayarlar, ardından hazırlanmış veri ve kayıp fonksiyonu ile `model.fit()` çağrısını yapar ve ince ayarlanmış modeli belirtilen çıktı yoluna kaydeder.

### Fine-tuned Model ile Embedding'leri Yenileme ve Test Etme
Fine-tuning sonrası, eğitim setinizin embedding'lerini yenilemeniz ve chatbot'unuzu yeni modelle test etmeniz gerekecek.

Bu markdown hücresi, sonraki adımların ince ayarlı modelle embedding'leri yeniden oluşturmayı ve güncellenmiş bu model kullanarak sohbet botunun performansını test etmeyi içereceğini belirtir.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity

# train_df'den soru ve cevap listelerini çıkar
train_questions = train_df['Question'].tolist()
train_answers = train_df['Answer'].tolist()

# Yeni eğitilmiş model ile eğitim setinin embedding'lerini yeniden oluştur
# Daha önce 'model' değişkeni ile eğitildiği için doğrudan kullanabiliriz.
print("Eğitim setinin embedding'leri yeniden encode ediliyor...")
fine_tuned_train_embeddings = model.encode(train_questions, show_progress_bar=True)
print("Tamamlandı! Yeni Embedding shape:", fine_tuned_train_embeddings.shape)

# Yeni embedding'leri kaydet
with open('fine_tuned_train_embeddings.pkl', 'wb') as f:
    pickle.dump(fine_tuned_train_embeddings, f)

print("Yeni embedding'ler kaydedildi.")

# Chatbot fonksiyonunu güncelleyerek yeni embedding'leri kullanmasını sağlayın
def chatbot_cevap_fine_tuned(kullanici_sorusu, benzerlik_esigi=0.70):
    normalized_soru = normalize_text(kullanici_sorusu) # Artık normalize_text kullanılıyor
    soru_embedding = model.encode([normalized_soru]) # Artık fine-tuned model kullanılıyor

    benzerlikler = cosine_similarity(soru_embedding, fine_tuned_train_embeddings)[0]
    en_benzer_idx = np.argmax(benzerlikler)
    benzerlik_skoru = benzerlikler[en_benzer_idx]

    if benzerlik_skoru < benzerlik_esigi:
        return f"Bu soruya uygun bir cevap bulunamadı (Benzerlik Skoru: {benzerlik_skoru:.2f}). Lütfen sorunuzu farklı şekilde ifade etmeyi deneyin veya daha spesifik olun."

    bulunan_soru = train_questions[en_benzer_idx]
    cevap = train_answers[en_benzer_idx]

    return f"**Benzer Soru:** {bulunan_soru}\n\n**Cevap:** {cevap}\n\n*Benzerlik Skoru: {benzerlik_skoru:.2f}"

Eğitim setinin embedding'leri yeniden encode ediliyor...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Tamamlandı! Yeni Embedding shape: (482, 384)
Yeni embedding'ler kaydedildi.


Bu hücre, ince ayarlanmış `model` kullanarak eğitim sorularını yeniden kodlar ve bu yeni embedding'leri kaydeder. Ayrıca, sohbet botunun ince ayarlanmış modeli kullanmasını sağlayarak, bu yeni embedding'leri ve ön işleme için `normalize_text` fonksiyonunu kullanmak üzere `chatbot_cevap_fine_tuned` fonksiyonunu günceller.

### Niyet Sınıflandırmasını Chatbot'a Entegre Etme

Şimdi eğitilmiş niyet sınıflandırıcısını, `chatbot_cevap_fine_tuned` fonksiyonunu kullanan ana sohbet botu arayüzümüze entegre edelim. Bu, kullanıcının sorusunun tahmin edilen niyetini de gösterecektir.

Bu markdown hücresi, niyet sınıflandırıcısının ana sohbet botu arayüzü `chatbot_cevap_fine_tuned` içine entegrasyonunu tanıtır ve kullanıcının sorusunun tahmin edilen niyetini cevabın yanında gösterir.

In [ ]:
def chatbot_with_intent(kullanici_sorusu, benzerlik_esigi=0.70):
    # Niyet tahmini için kullanıcının sorusunu encode et
    soru_embedding_intent = model.encode([normalize_text(kullanici_sorusu)])
    tahmin_edilen_niyet_encoded = intent_classifier.predict(soru_embedding_intent)[0]
    tahmin_edilen_niyet = label_encoder.inverse_transform([tahmin_edilen_niyet_encoded])[0]

    # Temel chatbot cevabını al
    chatbot_response = chatbot_cevap_fine_tuned(kullanici_sorusu, benzerlik_esigi)

    # Niyet bilgisini cevaba ekle
    return f"**Tahmin Edilen Niyet:** {tahmin_edilen_niyet}\n\n{chatbot_response}"

Bu hücre, niyet sınıflandırmasını içeren yeni bir sohbet botu fonksiyonu olan `chatbot_with_intent`'i tanımlar. İlk olarak, eğitilmiş `intent_classifier` kullanarak kullanıcının sorgusunun niyetini tahmin eder, ardından `chatbot_cevap_fine_tuned` fonksiyonundan bir yanıt alır ve son olarak tahmin edilen niyeti sohbet botunun cevabı ile birleştirir.

### Fine-tuned Model ile Değerlendirme

Şimdi `chatbot_cevap_fine_tuned` fonksiyonunu kullanarak modelin test seti üzerindeki performansını değerlendirelim.

Bu markdown hücresi, ince ayarlı modelin performansının `chatbot_cevap_fine_tuned` fonksiyonu kullanılarak değerlendirileceği bölümü gösteren bir başlık görevi görür.

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import nltk

smoother = SmoothingFunction().method1
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Skor listelerini sıfırla
bleu_scores_ft = []
rouge1_scores_ft = []
rouge2_scores_ft = []
rougeL_scores_ft = []
em_scores_ft = []
f1_scores_ft = []
similarity_scores_ft = []
tablo_satirlari_ft = []

print("Fine-tuned model ile test seti değerlendiriliyor (benzerlik eşiği = 0.3 ile)...")

for i, row in test_df.iterrows():
    soru = row['Question']
    referans_cevap = row['Answer']

    # Fine-tuned modelden tahmin al
    tahmin_full_output = chatbot_cevap_fine_tuned(soru, benzerlik_esigi=0.3)

    # Extract actual answer text from the chatbot's formatted output for evaluation
    actual_tahmin_text = ''
    if "Bu soruya uygun bir cevap bulunamadı." in tahmin_full_output:
        actual_tahmin_text = ""
    elif '**Cevap:**' in tahmin_full_output:
        start_index = tahmin_full_output.find('**Cevap:**') + len('**Cevap:**')
        end_index = tahmin_full_output.find('\n\n*Benzerlik Skoru:')
        if end_index != -1 and start_index != -1:
            actual_tahmin_text = tahmin_full_output[start_index:end_index].strip()
        else:
            actual_tahmin_text = tahmin_full_output.strip() # Fallback if format changes
    else:
        actual_tahmin_text = tahmin_full_output.strip()

    # BLEU
    if not actual_tahmin_text:
        bleu = 0.0
    else:
        referans_tokens = [referans_cevap.lower().split()]
        tahmin_tokens = actual_tahmin_text.lower().split()
        bleu = sentence_bleu(referans_tokens, tahmin_tokens, smoothing_function=smoother)
    bleu_scores_ft.append(bleu)

    # ROUGE
    if not actual_tahmin_text:
        rouge1_scores_ft.append(0.0)
        rouge2_scores_ft.append(0.0)
        rougeL_scores_ft.append(0.0)
    else:
        rouge_result = scorer.score(referans_cevap, actual_tahmin_text)
        rouge1_scores_ft.append(rouge_result['rouge1'].fmeasure)
        rouge2_scores_ft.append(rouge_result['rouge2'].fmeasure)
        rougeL_scores_ft.append(rouge_result['rougeL'].fmeasure)

    # Exact Match
    em_scores_ft.append(exact_match(referans_cevap, tahmin_full_output))

    # F1 Score
    f1_scores_ft.append(f1_score_hesapla(referans_cevap, tahmin_full_output))

    # Similarity Score
    similarity_start_index = tahmin_full_output.find('*Benzerlik Skoru:')
    if similarity_start_index != -1:
        similarity_score_str = tahmin_full_output[similarity_start_index + len('*Benzerlik Skoru:'):].strip()
        try:
            similarity_scores_ft.append(float(similarity_score_str))
        except ValueError:
            similarity_scores_ft.append(0.0)
    else:
        similarity_scores_ft.append(0.0)

    # Tablo için ilk 5 satırı kaydet
    if len(tablo_satirlari_ft) < 5:
        tablo_satirlari_ft.append({
            'Soru': soru,
            'Referans Cevap': referans_cevap[:100] + '...',
            'Model Cevabı': tahmin_full_output[:100] + '...' # Keep full output for display in table
        })


print("\n========== FINE-TUNED MODEL DEĞERLENDİRME SONUÇLARI ==========")
print(f"BLEU        : {sum(bleu_scores_ft)/len(bleu_scores_ft):.4f}")
print(f"ROUGE-1     : {sum(rouge1_scores_ft)/len(rouge1_scores_ft):.4f}")
print(f"ROUGE-2     : {sum(rouge2_scores_ft)/len(rouge2_scores_ft):.4f}")
print(f"ROUGE-L     : {sum(rougeL_scores_ft)/len(rougeL_scores_ft):.4f}")
print(f"Exact Match : {sum(em_scores_ft)/len(em_scores_ft):.4f}")
print(f"F1 Score    : {sum(f1_scores_ft)/len(f1_scores_ft):.4f}")
print(f"Similarity  : {sum(similarity_scores_ft)/len(similarity_scores_ft):.4f}")
print("==========================================================")

# Karşılaştırmalı tablo
tablo_df_ft = pd.DataFrame(tablo_satirlari_ft)
print("\n=== KARŞILAŞTIRMALI ÖRNEK TABLO (Fine-tuned) ===")
print(tablo_df_ft.to_string())

Fine-tuned model ile test seti değerlendiriliyor (benzerlik eşiği = 0.3 ile)...

========== FINE-TUNED MODEL DEĞERLENDİRME SONUÇLARI ==========
BLEU        : 0.0716
ROUGE-1     : 0.2055
ROUGE-2     : 0.0934
ROUGE-L     : 0.1673
Exact Match : 0.0385
F1 Score    : 0.1621
Similarity  : 0.5728

=== KARŞILAŞTIRMALI ÖRNEK TABLO (Fine-tuned) ===
                                               Soru                                                                                           Referans Cevap                                                                                               Model Cevabı
0                      how to stop taking trazodone  Do not stop taking trazodone without talking to your doctor. If you suddenly stop taking trazodone, ...  **Benzer Soru:** how to stop taking xanax\n\n**Cevap:** Because of the danger of withdrawal, abrupt di...
1            when is the best time to take lotensin  The morning administration is preferable because it more effectively covers th

Bu hücre, ince ayarlı sohbet botu modelini test setinde değerlendirir. Test setindeki her soru için BLEU, ROUGE-1, ROUGE-2, ROUGE-L, Tam Eşleşme (Exact Match), F1 Skoru ve Benzerlik Skoru gibi çeşitli metrikleri hesaplar. Ardından tüm metrikler için ortalama skorları yazdırır ve soru, referans cevap ve modelin ürettiği cevap ile ilk 5 örneğin karşılaştırmalı bir tablosunu görüntüler.

In [ ]:
import gradio as gr

def chatbot_arayuz(kullanici_sorusu):
    if not kullanici_sorusu.strip():
        return "Lütfen bir soru girin."
    return chatbot_cevap_fine_tuned(kullanici_sorusu) # Use the fine-tuned chatbot function

arayuz = gr.Interface(
    fn=chatbot_arayuz,
    inputs=gr.Textbox(
        label="Sorunuzu yazın",
        placeholder="Örnek: What is aspirin used for?",
        lines=2
    ),
    outputs=gr.Textbox(
        label="Cevap",
        lines=10
    ),
    title="💊 İlaç Bilgilendirme Chatbotu (Fine-tuned)", # Update title to reflect fine-tuning
    description="İlaçlarla ilgili sorularınızı İngilizce yazın.",
    examples=[
        ["What is morphine used for?"],
        ["What are the side effects of valium?"],
        ["Can I take aspirin with food?"]
    ]
)

arayuz.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5bb6428ec822963ddf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Bu hücre, ince ayarlı sohbet botu için bir Gradio arayüzü kurar ve başlatır. Kullanıcı girişini işlemek için `chatbot_arayuz`'u tanımlar ve yanıt almak için `chatbot_cevap_fine_tuned`'u çağırır. Arayüz, sorular için bir metin kutusu, yanıtlar için bir görüntüleme alanı, bir başlık, bir açıklama ve örnek sorular içerir.